# Phase 7 — Evaluation & Observability

**Localization eval** (SWE-bench-flavored): built self-supervised from commit history — question = commit subject, gold = the source files that commit changed. Measures whether retrieval surfaces the right files.

**Telemetry**: every `call_llm` records provider/model/latency/size (optional Langfuse mirror).

**Run first:** all prior phases indexed. Eval is retrieval-only (no LLM); the telemetry cell makes one Gemini call.

In [ ]:
import os, sys
from pathlib import Path

root = Path.cwd()
while root != root.parent and not (root / "pyproject.toml").exists():
    root = root.parent
os.chdir(root)
sys.path.insert(0, str(root / "src"))

from sqlalchemy import select
from archaeologist.models.db import session_scope
from archaeologist.models.entities import Repo
from archaeologist.eval import localization
from archaeologist.eval.dataset import build_localization_set
print("ready")

## Localization eval

In [ ]:
with session_scope() as s:
    repo = s.scalar(select(Repo))
    instances = build_localization_set(s, repo.id, limit=30, max_files=4)
print(f"{len(instances)} instances\n")

rows = localization.evaluate(instances)
agg = localization.aggregate(rows)
for k, v in agg.items():
    print(f"  {k:12}: {v}")

## Where it wins and loses
Descriptive changes localize; non-descriptive ones ('update dependencies') don't — the failure mode is uninformative queries, not broken retrieval.

In [ ]:
wins = [r for r in rows if r["hit@5"]]
misses = [r for r in rows if not r["hit@5"]]
print(f"hit@5: {len(wins)}/{len(rows)}\n")
print("WINS:")
for r in wins[:5]:
    print(f"  ✓ {r['question'][:50]:52} -> {r['gold']}")
print("\nMISSES:")
for r in misses[:5]:
    print(f"  ✗ {r['question'][:50]:52} -> {r['gold']}")

## Telemetry — LLM call instrumentation

In [ ]:
from archaeologist import telemetry
from archaeologist.rag.pipeline import answer_question

telemetry.reset()
answer_question("how are before_request handlers run", k=4)
answer_question("what does url_for do", k=4)
print("summary:", telemetry.summary())
for c in telemetry.calls():
    print(f"  {c['provider']}/{c['model']}  {c['latency_ms']}ms  in={c['in_chars']} out={c['out_chars']}")

## Summary

In [ ]:
print("Phase 7 — EVAL + OBSERVABILITY OK ✅")
print(f"  localization instances : {agg['n']}")
print(f"  hit@5 / hit@10         : {agg['hit@5']} / {agg['hit@10']}")
print(f"  MRR                    : {agg['MRR']}")
print("  telemetry              : per-call latency/size recorded (Langfuse optional)")